# Canonical correlation analysis: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cross_decomposition import CCA

rng = np.random.default_rng(10)

### 1. Two predictors (X1, X2) and two outcomes (Y1, Y2) linked through two hidden factors

In [ ]:
# 1. Two predictors (X1, X2) and two outcomes (Y1, Y2) linked through two hidden factors
n = 150
u1, u2 = rng.standard_normal(n), rng.standard_normal(n)
X = np.column_stack([u1 + 0.6 * rng.standard_normal(n), 0.5 * u1 + u2 + 0.6 * rng.standard_normal(n)])
Y = np.column_stack([1.0 * u1 + rng.standard_normal(n), 0.5 * u2 + rng.standard_normal(n)])
Rall = np.corrcoef(np.column_stack([X, Y]).T)
R11, R22, R12 = Rall[:2, :2], Rall[2:, 2:], Rall[:2, 2:]
print("best single pairwise correlation:", round(np.abs(R12).max(), 3))

### 2. Canonical correlations: square roots of the eigenvalues of R11^-1 R12 R22^-1 R21

In [ ]:
# 2. Canonical correlations: square roots of the eigenvalues of R11^-1 R12 R22^-1 R21
M = np.linalg.inv(R11) @ R12 @ np.linalg.inv(R22) @ R12.T
ev = np.sort(np.linalg.eigvals(M).real)[::-1]
rho = np.sqrt(ev)
print("canonical correlations:", np.round(rho, 3))

### 3. Check with sklearn: correlate the first pair of canonical variates

In [ ]:
# 3. Check with sklearn: correlate the first pair of canonical variates
Xs, Ys = (X - X.mean(0)) / X.std(0), (Y - Y.mean(0)) / Y.std(0)
cca = CCA(n_components=2, scale=False).fit(Xs, Ys)
U, V = cca.transform(Xs, Ys)
print("sklearn: corr(U1,V1) =", round(np.corrcoef(U[:, 0], V[:, 0])[0, 1], 3), "| corr(U2,V2) =", round(np.corrcoef(U[:, 1], V[:, 1])[0, 1], 3), "| corr(U1,U2) =", round(np.corrcoef(U[:, 0], U[:, 1])[0, 1], 3))

### 4. Bartlett tests (p = q = 2)

In [ ]:
# 4. Bartlett tests (p = q = 2)
p_, q_ = 2, 2
k = n - 1 - (p_ + q_ + 1) / 2
chi_all = -k * np.log(1 - rho ** 2).sum()
chi_2nd = -k * np.log(1 - rho[1] ** 2)
print("\nall correlations zero: chi2 =", round(chi_all, 2), "df 4, p =", format(stats.chi2.sf(chi_all, 4), ".2e"))
print("second correlation zero: chi2 =", round(chi_2nd, 2), "df 1, p =", round(stats.chi2.sf(chi_2nd, 1), 4))

### 5. One x and two y (the numerical problem): rho^2 is the R^2 of x on y1, y2

In [ ]:
# 5. One x and two y (the numerical problem): rho^2 is the R^2 of x on y1, y2
r1, r2, r12, nn = 0.5, 0.4, 0.2, 80
rho2 = (r1 ** 2 + r2 ** 2 - 2 * r1 * r2 * r12) / (1 - r12 ** 2)
Rxy = np.array([[r1, r2]]); Ryy = np.array([[1, r12], [r12, 1]])
print("\nrho^2 by formula:", round(rho2, 4), "| by matrices:", round((Rxy @ np.linalg.inv(Ryy) @ Rxy.T)[0, 0], 4))
chi = -(nn - 1 - 2) * np.log(1 - rho2)
print("Bartlett chi2:", round(chi, 3), "| 5% critical value (2 df):", round(stats.chi2.ppf(0.95, 2), 3))

### 6. Small samples overstate the association

In [ ]:
# 6. Small samples overstate the association
for m in (20, 50, 200, 1000):
    vals = []
    for _ in range(200):
        a, b = rng.standard_normal((m, 2)), rng.standard_normal((m, 2))
        Rz = np.corrcoef(np.column_stack([a, b]).T)
        Mz = np.linalg.inv(Rz[:2, :2]) @ Rz[:2, 2:] @ np.linalg.inv(Rz[2:, 2:]) @ Rz[2:, :2]
        vals.append(np.sqrt(np.linalg.eigvals(Mz).real.max()))
    print(f"independent sets, n={m:4d}: average first canonical correlation {np.mean(vals):.3f}")

## Expected output

Running all cells above should print:

```
best single pairwise correlation: 0.563
canonical correlations: [0.588 0.253]
sklearn: corr(U1,V1) = 0.588 | corr(U2,V2) = 0.253 | corr(U1,U2) = -0.0

all correlations zero: chi2 = 71.75 df 4, p = 9.71e-15
second correlation zero: chi2 = 9.67 df 1, p = 0.0019

rho^2 by formula: 0.3438 | by matrices: 0.3438
Bartlett chi2: 32.433 | 5% critical value (2 df): 5.991
independent sets, n=  20: average first canonical correlation 0.414
independent sets, n=  50: average first canonical correlation 0.253
independent sets, n= 200: average first canonical correlation 0.126
independent sets, n=1000: average first canonical correlation 0.056
```

## Practice exercises

Try these in the cells below.

1. Raise the link strength between X and Y and watch ρ₁ and the scatter of U₁ against V₁.
2. Set the link to zero and press New sample. Is the Bartlett test ever significant?
3. Make the variables within X highly correlated. Do the loadings stay stable between samples?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
